In [ ]:
# Это монтирует ваш Google Диск в среду Colab VM.
from google.colab import drive
drive.mount('/content/drive')

# TODO: Введите имя папки на Диске, где вы сохранили разархивированную
# папку заданий, например, 'cs231n/assignments/assignment2/'
FOLDERNAME = 'cs231n/assignments/assignment2/'
assert FOLDERNAME is not None, "[!] Введите имя папки."

# Теперь, когда мы смонтировали ваш Диск, это гарантирует, что
# интерпретатор Python в Colab VM может загружать
# файлы python из него.
import sys
sys.path.append('/content/drive/My Drive/{}'.format(FOLDERNAME))

# Это скачивает набор данных CIFAR-10 на ваш Диск,
# если он еще не существует.
%cd /content/drive/My\ Drive/$FOLDERNAME/cs231n/datasets/
!bash get_datasets.sh
%cd /content/drive/My\ Drive/$FOLDERNAME


# Описание Dropout и Batch Normalization
В этой главе мы изучаем две ключевые техники регуляризации, Dropout и Batch Normalization.
Обе эти техники используются для предотвращения переобучения модели.

Dropout — это простой, но очень эффективный метод. По сути, во время обучения Dropout "выключает" случайный набор нейронов (помеха), что заставляет сеть становиться более робастной.

Batch Normalization (BN) решает проблему внутренних ковариаций и стабилизирует обучение, нормализуя активации по мини-батчу.

Пример:
В классической сети без регуляризации, если какой-то признак или группа признаков оказывается сильно скоррелирована, сеть может начать чрезмерно полагаться только на эти признаки, что ведет к переобучению.
BN решает эту проблему, заставляя распределение активаций оставаться стабильным в процессе обучения, независимо от признаков, которые могут быть "засвечены" переобучением.

In [ ]:
# Настройка ячейки.
import time
import numpy as np
import matplotlib.pyplot as plt
from cs231n.data_utils import get_CIFAR10_data
from cs231n.gradient_check import eval_numerical_gradient, eval_numerical_gradient_array
from cs231n.solver import Solver

%matplotlib inline
plt.rcParams["figure.figsize"] = (10.0, 8.0)  # Установка размера диаграмм по умолчанию.
plt.rcParams["image.interpolation"] = "nearest"
plt.rcParams["image.cmap"] = "gray"

import sys
import types
import importlib

if "imp" not in sys.modules:
    imp = types.ModuleType("imp")
    imp.reload = importlib.reload
    sys.modules["imp"] = imp

%load_ext autoreload
%autoreload 2

def rel_error(x, y):
    """Возвращает относительную ошибку."""
    return np.max(np.abs(x - y) / (np.maximum(1e-8, np.abs(x) + np.abs(y))))

def print_mean_std(x,axis=0):
    print(f"  средние: {x.mean(axis=axis)}")
    print(f"  станд:  {x.std(axis=axis)}\n")

In [ ]:
# Load the (preprocessed) CIFAR-10 data.
data = get_CIFAR10_data()
for k, v in list(data.items()):
    print(f"{k}: {v.shape}")

# Нормализация мини-батчей: Прямой проход (Batch Normalization: Forward Pass)
В файле `cs231n/layers.py` реализуйте прямой проход нормализации мини-батчей в функции `batchnorm_forward`. Как только вы это сделаете, запустите следующее для тестирования вашей реализации.

Обращение к статье, связанной выше в [1], может быть полезным!


In [ ]:
from cs231n.layers import *

# Проверка прямого прохода во время обучения путем проверки средних значений и дисперсий
# признаков до и после пакетной нормализации

# Моделирование прямого прохода для двухслойной сети.
np.random.seed(231)
N, D1, D2, D3 = 200, 50, 60, 3
X = np.random.randn(N, D1)
W1 = np.random.randn(D1, D2)
W2 = np.random.randn(D2, D3)
a = np.maximum(0, X.dot(W1)).dot(W2)

print('Before batch normalization:')
print_mean_std(a,axis=0)

gamma = np.ones((D3,))
beta = np.zeros((D3,))

# Средние значения должны быть близки к нулю, а стандартные отклонения близки к единице.
print('After batch normalization (gamma=1, beta=0)')
a_norm, _ = batchnorm_forward(a, gamma, beta, {'mode': 'train'})
print_mean_std(a_norm,axis=0)

gamma = np.asarray([1.0, 2.0, 3.0])
beta = np.asarray([11.0, 12.0, 13.0])

# Теперь средние значения должны быть близки к beta, а стандартные отклонения близки к gamma.
print('After batch normalization (gamma=', gamma, ', beta=', beta, ')')
a_norm, _ = batchnorm_forward(a, gamma, beta, {'mode': 'train'})
print_mean_std(a_norm,axis=0)

In [ ]:
# Проверка прямого прохода во время тестирования 
# Выполняется путем многократного выполнения
# прямого прохода в режиме обучения для прогрева скользящих средних, а затем
# проверки средних значений и дисперсий активаций после прямого прохода в режиме тестирования.

np.random.seed(231)
N, D1, D2, D3 = 200, 50, 60, 3
W1 = np.random.randn(D1, D2)
W2 = np.random.randn(D2, D3)

bn_param = {'mode': 'train'}
gamma = np.ones(D3)
beta = np.zeros(D3)

for t in range(50):
  X = np.random.randn(N, D1)
  a = np.maximum(0, X.dot(W1)).dot(W2)
  batchnorm_forward(a, gamma, beta, bn_param)

bn_param['mode'] = 'test'
X = np.random.randn(N, D1)
a = np.maximum(0, X.dot(W1)).dot(W2)
a_norm, _ = batchnorm_forward(a, gamma, beta, bn_param)

# Средние значения должны быть близки к нулю, а стандартные отклонения близки к единице,
# но будут более шумными, чем при прямых проходах во время обучения.
print('After batch normalization (test-time):')
print_mean_std(a_norm,axis=0)

# Пакетная нормализация: обратный проход

Теперь реализуйте обратный проход для пакетной нормализации в функции `batchnorm_backward`.

Для вывода обратного прохода следует выписать граф вычислений для пакетной нормализации и выполнить обратное распространение через каждый из промежуточных узлов. Некоторые промежуточные узлы могут иметь несколько исходящих ветвей; обязательно просуммируйте градиенты по этим ветвям при обратном проходе. Может быть полезно обратиться к статье, ссылка на которую приведена выше в [1].

Закончив, запустите следующий код, чтобы численно проверить корректность обратного прохода.

_Подсказка: [здесь](https://www.adityaagrawal.net/blog/deep_learning/bprop_batch_norm) есть полезный материал, объясняющий, как градиенты выводятся из статьи._

In [ ]:
# проверка градиента для обратного прохода пакетной нормализации.
np.random.seed(231)
N, D = 4, 5
x = 5 * np.random.randn(N, D) + 12
gamma = np.random.randn(D)
beta = np.random.randn(D)
dout = np.random.randn(N, D)

bn_param = {'mode': 'train'}
fx = lambda x: batchnorm_forward(x, gamma, beta, bn_param)[0]
fg = lambda a: batchnorm_forward(x, a, beta, bn_param)[0]
fb = lambda b: batchnorm_forward(x, gamma, b, bn_param)[0]

dx_num = eval_numerical_gradient_array(fx, x, dout)
da_num = eval_numerical_gradient_array(fg, gamma.copy(), dout)
db_num = eval_numerical_gradient_array(fb, beta.copy(), dout)

_, cache = batchnorm_forward(x, gamma, beta, bn_param)
dx, dgamma, dbeta = batchnorm_backward(dout, cache)

# Вы должны ожидать относительные ошибки между 1e-13 и 1e-8.
print('dx error: ', rel_error(dx_num, dx))
print('dgamma error: ', rel_error(da_num, dgamma))
print('dbeta error: ', rel_error(db_num, dbeta))

# Пакетная нормализация: альтернативный обратный проход

На лекции мы рассматривали две разные реализации обратного прохода для сигмоиды.  
Одна стратегия — выписать граф вычислений, состоящий из простых операций, и выполнить обратное распространение через все промежуточные значения.  
Другая стратегия — вывести производные на бумаге.  
Например, для обратного прохода функции сигмоиды можно получить очень простую формулу, упростив градиенты аналитически.

Оказывается, аналогичное упрощение можно сделать и для обратного прохода пакетной нормализации!

В прямом проходе, получив набор входов $X=\begin{bmatrix}x_1\\x_2\\...\\x_N\end{bmatrix}$ ,

сначала вычисляются среднее $\mu$ и дисперсия $v$.
Зная $\mu$ и $v$, можно вычислить стандартное отклонение $\sigma$ и нормализованные данные $Y$.  

Приведённые ниже уравнения и графическая иллюстрация описывают вычисления ($y_i$ — это $i$-й элемент вектора $Y$).

\begin{align}
& \mu=\frac{1}{N}\sum_{k=1}^N x_k  &  v=\frac{1}{N}\sum_{k=1}^N (x_k-\mu)^2 \\
& \sigma=\sqrt{v+\epsilon}         &  y_i=\frac{x_i-\mu}{\sigma}
\end{align}

<img src="https://raw.githubusercontent.com/cs231n/cs231n.github.io/master/assets/a2/batchnorm_graph.png">

Суть задачи при обратном распространении ошибки — вычислить $\frac{\partial L}{\partial X}$, используя поступающий градиент сверху $\frac{\partial L}{\partial Y}$. Для этого вспомним, что по правилу дифференцирования сложной функции $\frac{\partial L}{\partial X} = \frac{\partial L}{\partial Y} \cdot \frac{\partial Y}{\partial X}$.

Неизвестная и сложная часть — это $\frac{\partial Y}{\partial X}$. Её можно найти, сначала пошагово выведя локальные градиенты:  
$\frac{\partial v}{\partial X}$, $\frac{\partial \mu}{\partial X}$,  
$\frac{\partial \sigma}{\partial v}$,  
$\frac{\partial Y}{\partial \sigma}$ и $\frac{\partial Y}{\partial \mu}$,  
а затем с помощью правила дифференцирования сложной функции соответствующим образом композиционно объединить эти градиенты (которые имеют вид векторов!), чтобы вычислить $\frac{\partial Y}{\partial X}$.

Напрямую работать с градиентами по $X$ и $Y$, требующими матричного умножения может показаться вам сложно.  
Попробуйте сначала рассуждать в терминах отдельных элементов $x_i$ и $y_i$: в этом случае вам потребуется вывести выражения для $\frac{\partial L}{\partial x_i}$, используя правило дифференцирования сложной функции для промежуточных вычислений $\frac{\partial \mu}{\partial x_i}$, $\frac{\partial v}{\partial x_i}$, $\frac{\partial \sigma}{\partial x_i}$, а затем собрать эти компоненты воедино, чтобы получить $\frac{\partial y_i}{\partial x_i}$.

Убедитесь, что все промежуточные выкладки градиентов максимально упрощены — это облегчит реализацию.

После этого реализуйте упрощённый обратный проход пакетной нормализации в функции `batchnorm_backward_alt` и сравните две реализации, запустив следующий код. Обе реализации должны давать практически идентичные результаты, но альтернативная должна работать немного быстрее.

_Подсказка: https://cs.stanford.edu/people/jcjohns/batchnorm.pdf. Обратите внимание, что $\gamma$ должен быть учтён в итоговом выводе._

_Обратите внимание, что уравнение (8) в этом конспекте должно иметь вид:_  
$$
\frac{1}{\sigma} \frac{\partial}{\partial x_i} (x_j - \mu) - \frac{1}{\sigma^2} \frac{\partial \sigma}{\partial x_i} (x_j - \mu)
$$

In [ ]:
np.random.seed(231)
N, D = 100, 500
x = 5 * np.random.randn(N, D) + 12
gamma = np.random.randn(D)
beta = np.random.randn(D)
dout = np.random.randn(N, D)

bn_param = {'mode': 'train'}
out, cache = batchnorm_forward(x, gamma, beta, bn_param)

t1 = time.time()
dx1, dgamma1, dbeta1 = batchnorm_backward(dout, cache)
t2 = time.time()
dx2, dgamma2, dbeta2 = batchnorm_backward_alt(dout, cache)
t3 = time.time()

print('dx difference: ', rel_error(dx1, dx2))
print('dgamma difference: ', rel_error(dgamma1, dgamma2))
print('dbeta difference: ', rel_error(dbeta1, dbeta2))
print('speedup: %.2fx' % ((t2 - t1) / (t3 - t2)))

# Полносвязные сети с нормализацией батча

Теперь, когда у вас есть рабочая реализация нормализации батча, вернитесь к вашей реализации `FullyConnectedNet` в файле `cs231n/classifiers/fc_net.py`. Напомним, что вы реализовали инициализацию сети, прямой проход и обратный проход в задании 1. Скопируйте эту реализацию сюда и модифицируйте её, чтобы добавить пакетную нормализацию.

В частности, если в конструкторе флаг `normalization` установлен в значение `"batchnorm"`, вы должны вставить слой пакетной нормализации перед каждой нелинейностью ReLU. Выход последнего слоя сети нормализовать не нужно. Закончив, запустите следующий код для проверки градиентов вашей реализации.

**Подсказка:** Возможно, вам будет полезно определить дополнительный вспомогательный слой, аналогичный тем, что приведены в файле `cs231n/layer_utils.py`.

In [ ]:
from cs231n.classifiers.fc_net import *
from cs231n.gradient_check import *

np.random.seed(231)
N, D, H1, H2, C = 2, 15, 20, 30, 10
X = np.random.randn(N, D)
y = np.random.randint(C, size=(N,))

# Расхождение в градиентах 1e-4~1e-10 for W,
# between 1e-08~1e-10 for b,
# and between 1e-08~1e-09 for beta and gammas.
for reg in [0, 3.14]:
  print('Running check with reg = ', reg)
  model = FullyConnectedNet([H1, H2], input_dim=D, num_classes=C,
                            reg=reg, weight_scale=5e-2, dtype=np.float64,
                            normalization='batchnorm')

  loss, grads = model.loss(X, y)
  print('Initial loss: ', loss)

  for name in sorted(grads):
    f = lambda _: model.loss(X, y)[0]
    grad_num = eval_numerical_gradient(f, model.params[name], verbose=False, h=1e-5)
    print('%s relative error: %.2e' % (name, rel_error(grad_num, grads[name])))
  if reg == 0: print()

# Пакетная нормализация для глубоких сетей

Запустите следующий код, чтобы обучить шестислойную сеть на подмножестве из 1000 обучающих примеров — как с пакетной нормализацией (Batch Normalization), так и без неё.

In [ ]:
np.random.seed(231)

# Попробуйте обучить глубокую сеть с пакетной нормализацией batchnorm.
hidden_dims = [100, 100, 100, 100, 100]

num_train = 1000
small_data = {
  'X_train': data['X_train'][:num_train],
  'y_train': data['y_train'][:num_train],
  'X_val': data['X_val'],
  'y_val': data['y_val'],
}

weight_scale = 2e-2
bn_model = FullyConnectedNet(hidden_dims, weight_scale=weight_scale, normalization='batchnorm')
model = FullyConnectedNet(hidden_dims, weight_scale=weight_scale, normalization=None)

print('Solver with batch norm:')
bn_solver = Solver(bn_model, small_data,
                num_epochs=10, batch_size=50,
                update_rule='adam',
                optim_config={
                  'learning_rate': 1e-3,
                },
                verbose=True,print_every=20)
bn_solver.train()

print('\nSolver without batch norm:')
solver = Solver(model, small_data,
                num_epochs=10, batch_size=50,
                update_rule='adam',
                optim_config={
                  'learning_rate': 1e-3,
                },
                verbose=True, print_every=20)
solver.train()

Запустите следующий код, чтобы визуализировать результаты работы двух обученных выше сетей.  
Вы должны показать, что использование пакетной нормализации (Batch Normalization) помогает сети сходиться значительно быстрее.

In [ ]:
def plot_training_history(title, label, baseline, bn_solvers, plot_fn, bl_marker='.', bn_marker='.', labels=None):
    """utility function for plotting training history"""
    plt.title(title)
    plt.xlabel(label)
    bn_plots = [plot_fn(bn_solver) for bn_solver in bn_solvers]
    bl_plot = plot_fn(baseline)
    num_bn = len(bn_plots)
    for i in range(num_bn):
        label='with_norm'
        if labels is not None:
            label += str(labels[i])
        plt.plot(bn_plots[i], bn_marker, label=label)
    label='baseline'
    if labels is not None:
        label += str(labels[0])
    plt.plot(bl_plot, bl_marker, label=label)
    plt.legend(loc='lower center', ncol=num_bn+1)


plt.subplot(3, 1, 1)
plot_training_history('Training loss','Iteration', solver, [bn_solver], \
                      lambda x: x.loss_history, bl_marker='o', bn_marker='o')
plt.subplot(3, 1, 2)
plot_training_history('Training accuracy','Epoch', solver, [bn_solver], \
                      lambda x: x.train_acc_history, bl_marker='-o', bn_marker='-o')
plt.subplot(3, 1, 3)
plot_training_history('Validation accuracy','Epoch', solver, [bn_solver], \
                      lambda x: x.val_acc_history, bl_marker='-o', bn_marker='-o')

plt.gcf().set_size_inches(15, 15)
plt.show()

# Пакетная нормализация и инициализация весов

Теперь мы проведём небольшой эксперимент, чтобы изучить взаимодействие пакетной нормализации (Batch Normalization) и инициализации весов.

В первой ячейке будут обучены восьмислойные сети — как с пакетной нормализацией, так и без неё — с разными масштабами инициализации весов. Во второй ячейке будут построены графики зависимости точности на обучении, точности на валидационном наборе и потерь на обучении от масштаба инициализации весов.

In [ ]:
np.random.seed(231)

# Исследуйте как влияют разные масштабы начальных значений весов W на обучение глубокой сети
hidden_dims = [50, 50, 50, 50, 50, 50, 50]
num_train = 1000
small_data = {
  'X_train': data['X_train'][:num_train],
  'y_train': data['y_train'][:num_train],
  'X_val': data['X_val'],
  'y_val': data['y_val'],
}

bn_solvers_ws = {}
solvers_ws = {}
weight_scales = np.logspace(-4, 0, num=20)
for i, weight_scale in enumerate(weight_scales):
    print('Running weight scale %d / %d' % (i + 1, len(weight_scales)))
    bn_model = FullyConnectedNet(hidden_dims, weight_scale=weight_scale, normalization='batchnorm')
    model = FullyConnectedNet(hidden_dims, weight_scale=weight_scale, normalization=None)

    bn_solver = Solver(bn_model, small_data,
                  num_epochs=10, batch_size=50,
                  update_rule='adam',
                  optim_config={
                    'learning_rate': 1e-3,
                  },
                  verbose=False, print_every=200)
    bn_solver.train()
    bn_solvers_ws[weight_scale] = bn_solver

    solver = Solver(model, small_data,
                  num_epochs=10, batch_size=50,
                  update_rule='adam',
                  optim_config={
                    'learning_rate': 1e-3,
                  },
                  verbose=False, print_every=200)
    solver.train()
    solvers_ws[weight_scale] = solver

In [ ]:
# Визуализация результатов эксперимента
best_train_accs, bn_best_train_accs = [], []
best_val_accs, bn_best_val_accs = [], []
final_train_loss, bn_final_train_loss = [], []

for ws in weight_scales:
  best_train_accs.append(max(solvers_ws[ws].train_acc_history))
  bn_best_train_accs.append(max(bn_solvers_ws[ws].train_acc_history))

  best_val_accs.append(max(solvers_ws[ws].val_acc_history))
  bn_best_val_accs.append(max(bn_solvers_ws[ws].val_acc_history))

  final_train_loss.append(np.mean(solvers_ws[ws].loss_history[-100:]))
  bn_final_train_loss.append(np.mean(bn_solvers_ws[ws].loss_history[-100:]))

plt.subplot(3, 1, 1)
plt.title('Best val accuracy vs. weight initialization scale')
plt.xlabel('Weight initialization scale')
plt.ylabel('Best val accuracy')
plt.semilogx(weight_scales, best_val_accs, '-o', label='baseline')
plt.semilogx(weight_scales, bn_best_val_accs, '-o', label='batchnorm')
plt.legend(ncol=2, loc='lower right')

plt.subplot(3, 1, 2)
plt.title('Best train accuracy vs. weight initialization scale')
plt.xlabel('Weight initialization scale')
plt.ylabel('Best training accuracy')
plt.semilogx(weight_scales, best_train_accs, '-o', label='baseline')
plt.semilogx(weight_scales, bn_best_train_accs, '-o', label='batchnorm')
plt.legend()

plt.subplot(3, 1, 3)
plt.title('Final training loss vs. weight initialization scale')
plt.xlabel('Weight initialization scale')
plt.ylabel('Final training loss')
plt.semilogx(weight_scales, final_train_loss, '-o', label='baseline')
plt.semilogx(weight_scales, bn_final_train_loss, '-o', label='batchnorm')
plt.legend()
plt.gca().set_ylim(1.0, 3.5)

plt.gcf().set_size_inches(15, 15)
plt.show()

## Вопрос 1:  
Опишите результаты этого эксперимента. Как масштаб инициализации весов по‑разному влияет на модели с пакетной нормализацией и без неё — и почему?

## Ответ:  
[ЗАПОЛНИТЕ ЗДЕСЬ]

# Пакетная нормализация и размер батча

Теперь мы проведём небольшой эксперимент, чтобы изучить взаимодействие пакетной нормализации (Batch Normalization) и размера батча.

В первой ячейке будут обучены шестислойные сети — как с пакетной нормализацией, так и без неё — с разными размерами батчей.  
Во второй ячейке будут построены графики зависимости точности на обучении и точности на валидации от времени.

In [ ]:
def run_batchsize_experiments(normalization_mode):
    np.random.seed(231)

    # Try training a very deep net with batchnorm.
    hidden_dims = [100, 100, 100, 100, 100]
    num_train = 1000
    small_data = {
      'X_train': data['X_train'][:num_train],
      'y_train': data['y_train'][:num_train],
      'X_val': data['X_val'],
      'y_val': data['y_val'],
    }
    n_epochs=10
    weight_scale = 2e-2
    batch_sizes = [5,10,50]
    lr = 10**(-3.5)
    solver_bsize = batch_sizes[0]

    print('No normalization: batch size = ',solver_bsize)
    model = FullyConnectedNet(hidden_dims, weight_scale=weight_scale, normalization=None)
    solver = Solver(model, small_data,
                    num_epochs=n_epochs, batch_size=solver_bsize,
                    update_rule='adam',
                    optim_config={
                      'learning_rate': lr,
                    },
                    verbose=False)
    solver.train()

    bn_solvers = []
    for i in range(len(batch_sizes)):
        b_size=batch_sizes[i]
        print('Normalization: batch size = ',b_size)
        bn_model = FullyConnectedNet(hidden_dims, weight_scale=weight_scale, normalization=normalization_mode)
        bn_solver = Solver(bn_model, small_data,
                        num_epochs=n_epochs, batch_size=b_size,
                        update_rule='adam',
                        optim_config={
                          'learning_rate': lr,
                        },
                        verbose=False)
        bn_solver.train()
        bn_solvers.append(bn_solver)

    return bn_solvers, solver, batch_sizes

batch_sizes = [5,10,50]
bn_solvers_bsize, solver_bsize, batch_sizes = run_batchsize_experiments('batchnorm')

In [ ]:
plt.subplot(2, 1, 1)
plot_training_history('Training accuracy (Batch Normalization)','Epoch', solver_bsize, bn_solvers_bsize, \
                      lambda x: x.train_acc_history, bl_marker='-^', bn_marker='-o', labels=batch_sizes)
plt.subplot(2, 1, 2)
plot_training_history('Validation accuracy (Batch Normalization)','Epoch', solver_bsize, bn_solvers_bsize, \
                      lambda x: x.val_acc_history, bl_marker='-^', bn_marker='-o', labels=batch_sizes)

plt.gcf().set_size_inches(15, 10)
plt.show()

## Вопрос 2:  
Опишите результаты этого эксперимента. Что они говорят о связи между пакетной нормализацией (Batch Normalization) и размером батча? Почему наблюдается такая зависимость?

## Ответ:  
[ЗАПОЛНИТЕ]


# Dropout

Dropout [1] — это метод регуляризации нейронных сетей, при котором во время прямого прохода некоторые выходные активации случайно обнуляются. В этом упражнении вам предстоит реализовать слой Dropout и модифицировать свою полносвязную сеть, чтобы она могла опционально использовать Dropout.

[1] [Geoffrey E. Hinton et al, "Improving neural networks by preventing co-adaptation of feature detectors", arXiv 2012](https://arxiv.org/abs/1207.0580)

In [ ]:
# Setup cell.
import time
import numpy as np
import matplotlib.pyplot as plt
from cs231n.classifiers.fc_net import *
from cs231n.data_utils import get_CIFAR10_data
from cs231n.gradient_check import eval_numerical_gradient, eval_numerical_gradient_array
from cs231n.solver import Solver

%matplotlib inline
plt.rcParams["figure.figsize"] = (10.0, 8.0)  # Set default size of plots.
plt.rcParams["image.interpolation"] = "nearest"
plt.rcParams["image.cmap"] = "gray"

import sys
import types
import importlib

if "imp" not in sys.modules:
    imp = types.ModuleType("imp")
    imp.reload = importlib.reload
    sys.modules["imp"] = imp

%load_ext autoreload
%autoreload 2

def rel_error(x, y):
    """Returns relative error."""
    return np.max(np.abs(x - y) / (np.maximum(1e-8, np.abs(x) + np.abs(y))))

In [ ]:
# Load the (preprocessed) CIFAR-10 data.
data = get_CIFAR10_data()
for k, v in list(data.items()):
    print(f"{k}: {v.shape}")

# Dropout: прямой проход

В файле `cs231n/layers.py` реализуйте прямой проход для Dropout. Поскольку поведение Dropout различается на этапе обучения и тестирования, обязательно реализуйте операцию для обоих режимов.

После этого запустите ячейку ниже, чтобы протестировать свою реализацию.

In [ ]:
np.random.seed(231)
x = np.random.randn(500, 500) + 10

for p in [0.25, 0.4, 0.7]:
    out, _ = dropout_forward(x, {'mode': 'train', 'p': p})
    out_test, _ = dropout_forward(x, {'mode': 'test', 'p': p})

    print('Running tests with p = ', p)
    print('Mean of input: ', x.mean())
    print('Mean of train-time output: ', out.mean())
    print('Mean of test-time output: ', out_test.mean())
    print('Fraction of train-time output set to zero: ', (out == 0).mean())
    print('Fraction of test-time output set to zero: ', (out_test == 0).mean())
    print()

# Dropout: обратный проход

В файле `cs231n/layers.py` реализуйте обратный проход для Dropout. После этого запустите следующую ячейку, чтобы численно проверить корректность вашей реализации.

In [ ]:
np.random.seed(231)
x = np.random.randn(10, 10) + 10
dout = np.random.randn(*x.shape)

dropout_param = {'mode': 'train', 'p': 0.2, 'seed': 123}
out, cache = dropout_forward(x, dropout_param)
dx = dropout_backward(dout, cache)
dx_num = eval_numerical_gradient_array(lambda xx: dropout_forward(xx, dropout_param)[0], x, dout)

# Error should be around e-10 or less.
print('dx relative error: ', rel_error(dx, dx_num))

## Вопрос 3:  
Что произойдёт, если не делить значения, проходящие через обратный Dropout, на `p` в слое Dropout? Почему это происходит?

## Ответ:  
[ЗАПОЛНИТЕ ЭТО]


# Полносвязные сети с Dropout

В файле `cs231n/classifiers/fc_net.py` измените реализацию, чтобы использовать Dropout.  
В частности, если конструктор сети получает значение параметра `dropout_keep_ratio`, отличное от 1, то сеть должна добавлять слой Dropout сразу после каждой нелинейности ReLU.  
После этого запустите следующий код для численной проверки градиентов вашей реализации.

In [ ]:
np.random.seed(231)
N, D, H1, H2, C = 2, 15, 20, 30, 10
X = np.random.randn(N, D)
y = np.random.randint(C, size=(N,))

for dropout_keep_ratio in [1, 0.75, 0.5]:
    print('Running check with dropout = ', dropout_keep_ratio)
    model = FullyConnectedNet(
        [H1, H2],
        input_dim=D,
        num_classes=C,
        weight_scale=5e-2,
        dtype=np.float64,
        dropout_keep_ratio=dropout_keep_ratio,
        seed=123
    )

    loss, grads = model.loss(X, y)
    print('Initial loss: ', loss)

    # Relative errors should be around e-6 or less.
    # Note that it's fine if for dropout_keep_ratio=1 you have W2 error be on the order of e-5.
    for name in sorted(grads):
        f = lambda _: model.loss(X, y)[0]
        grad_num = eval_numerical_gradient(f, model.params[name], verbose=False, h=1e-5)
        print('%s relative error: %.2e' % (name, rel_error(grad_num, grads[name])))
    print()

# Эксперимент с регуляризацией

В качестве эксперимента мы обучим пару двухслойных сетей на 500 обучающих примерах: в одной Dropout использоваться не будет, а в другой вероятность сохранения нейронов (keep probability) составит 0,25. Затем мы визуализируем динамику точности на обучающей и валидационной выборках для обеих сетей.

In [ ]:
# Обучите две сети, с dropout и без него
np.random.seed(231)
num_train = 500
small_data = {
    'X_train': data['X_train'][:num_train],
    'y_train': data['y_train'][:num_train],
    'X_val': data['X_val'],
    'y_val': data['y_val'],
}

solvers = {}
dropout_choices = [1, 0.25]
for dropout_keep_ratio in dropout_choices:
    model = FullyConnectedNet(
        [500],
        dropout_keep_ratio=dropout_keep_ratio
    )
    print(dropout_keep_ratio)

    solver = Solver(
        model,
        small_data,
        num_epochs=25,
        batch_size=100,
        update_rule='adam',
        optim_config={'learning_rate': 5e-4,},
        verbose=True,
        print_every=100
    )
    solver.train()
    solvers[dropout_keep_ratio] = solver
    print()

In [ ]:
# Plot train and validation accuracies of the two models.
train_accs = []
val_accs = []
for dropout_keep_ratio in dropout_choices:
    solver = solvers[dropout_keep_ratio]
    train_accs.append(solver.train_acc_history[-1])
    val_accs.append(solver.val_acc_history[-1])

plt.subplot(3, 1, 1)
for dropout_keep_ratio in dropout_choices:
    plt.plot(
        solvers[dropout_keep_ratio].train_acc_history, 'o', label='%.2f dropout_keep_ratio' % dropout_keep_ratio)
plt.title('Train accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(ncol=2, loc='lower right')

plt.subplot(3, 1, 2)
for dropout_keep_ratio in dropout_choices:
    plt.plot(
        solvers[dropout_keep_ratio].val_acc_history, 'o', label='%.2f dropout_keep_ratio' % dropout_keep_ratio)
plt.title('Val accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(ncol=2, loc='lower right')

plt.gcf().set_size_inches(15, 15)
plt.show()

## Вопрос 4:  
Сравните точность на валидационной и обучающей выборках с Dropout и без него — что ваши результаты позволяют сказать о Dropout как о средстве регуляризации?

## Ответ:  
[ЗАПОЛНИТЕ]
